

### 02 - Data Preparation

#### Purpose

- This notebook prepares the IBM Telco Customer Churn dataset for neural-network training with PyTorch.

- The goal is to convert the existing Gold Delta table into neural-network-ready batches.

- The complete data flow is:

``` text

Gold Delta Table
      ↓
Spark DataFrame
      ↓
Pandas DataFrame
      ↓
Create X and y
      ↓
Train / Validation / Test Split
      ↓
Preprocessing
      │
      ├── Numerical Features
      │     ├── Missing Value Imputation
      │     └── Standardization
      │
      ├── Categorical Features
      │     └── One-Hot Encoding
      │
      └── Binary Features
            └── Passthrough
      ↓
NumPy Arrays
      ↓
PyTorch Tensors
      ↓
TensorDataset
      ↓
DataLoader
      ↓
Neural-Network-Ready Batches

```


##### Production Design

This notebook is responsible for:

- Loading the curated Telco Gold dataset
- Creating train / validation / test splits
- Applying training-only preprocessing
- Verifying prepared feature matrices
- Converting prepared data into PyTorch tensors and DataLoaders

Reusable preprocessing logic is separated from notebook orchestration so the same transformations can later be reused consistently for:

- Model training
- Validation
- Testing
- Batch inference
- Production inference


##### Technologies Used

- Databricks
- Delta Lake
- Apache Spark
- Pandas
- NumPy
- Scikit-learn
- PyTorch

Key preprocessing components:

- train_test_split
- ColumnTransformer
- Pipeline
- SimpleImputer
- StandardScaler
- OneHotEncoder
- PyTorch Tensor
- TensorDataset
- DataLoader


##### Input

Gold Delta table: dbw_agentic_ai_dev.telco_ai.gold_telco

The Gold table contains:

- Customer demographic information
- Service information
- Contract information
- Billing information
- Churn
- Churn_Flag
- Tenure_Group

Dataset size: 7,043 customers


##### Output

This notebook produces:

- X_train_tensor
- y_train_tensor
- X_val_tensor
- y_val_tensor
- X_test_tensor
- y_test_tensor

and:

- train_dataset
- val_dataset
- test_dataset

and:

- train_loader
- val_loader
- test_loader

Final training batch shape:

X_batch → [32, 45]

y_batch → [32, 1]

This means:

- 32 customers per batch
- 45 prepared input features per customer
- 1 churn target per customer


##### Architecture

``` text

Gold Delta Table
      ↓
Spark DataFrame
      ↓
Pandas DataFrame
      ↓
Remove Non-Model Columns
      ↓
Create X and y
      ↓
Identify Feature Types
      │
      ├── Numerical
      ├── Binary
      └── Categorical
      ↓
Train / Validation / Test Split
      ↓
Fit Preprocessor on X_train Only
      ↓
Transform Train / Validation / Test
      ↓
Numeric Feature Matrix
      ↓
PyTorch Tensors
      ↓
TensorDataset
      ↓
DataLoader
      ↓
Batches
      ↓
Ready for Neural Network

```

In [0]:
%run ./00_project_config


##### 1. Load the Gold Delta Table

In [0]:
# Load the Gold table
df_spark = spark.table(GOLD_TABLE)

# inspect the schema
df_spark.printSchema()

display(df_spark.limit(5))


The Gold table contains both model features and columns that should not be used directly as neural-network inputs.

Examples:

customerID
→ identifier only

Churn
→ original string target

Churn_Flag
→ numerical target

Tenure_Group
→ derived from tenure

##### 2. Convert Spark DataFrame to Pandas

In [0]:
df = df_spark.toPandas()

print("Dataset shape:", df.shape)
display(df.head())

df.columns.tolist()


After `.toPandas()`:

df
→ Pandas DataFrame

The preprocessing workflow will use Pandas and scikit-learn before converting the processed data into PyTorch tensors.


##### 3. Define the Target

In [0]:

y = df[TARGET_COLUMN]

display(y.head())


The target represents:

- 0 → No Churn
- 1 → Churn

This target will later become:

- y_train
- y_val
- y_test

and eventually:

y_batch


##### 4. Create Input Features X

In [0]:
# Exclude columns that should not be used as model inputs:

X = df.drop(
    columns=COLUMNS_TO_DROP,
    errors="ignore",
)

In [0]:
print("X shape:", X.shape)
print("y shape:", y.shape)


Tenure_Group is excluded because it was derived from tenure.

Keeping the original continuous tenure feature allows the neural network to learn relationships directly without manually grouping tenure into buckets.


##### 5. Inspect the Target Distribution

In [0]:
print("Counts:")
print(y.value_counts())

print("\nProportions:")
print(y.value_counts(normalize=True))


The dataset contains some class imbalance.

This does not require immediate modification.

The imbalance will later be considered when evaluating:

- Precision
- Recall
- F1 Score
- ROC-AUC
- Classification Threshold
- Potential Class Weighting


##### 6. Inspect Data Types

In [0]:
X.dtypes

In [0]:

#Numerical columns:

X.select_dtypes(
    include=["number"]
).columns.tolist()

In [0]:
# Categorical columns:

X.select_dtypes(
    include=["object", "category", "bool"]
).columns.tolist()


Data type alone does not fully determine preprocessing.

SeniorCitizen is technically numeric, but semantically it is already a binary indicator:

- 0 → No
- 1 → Yes

Therefore it will be treated separately from continuous numerical features.

##### 7. Verify Configured Feature Groups

In [0]:
#Notice that we're no longer asking Pandas to decide how the model should treat the columns.We have made that modeling decision ourselves.

print("Numerical features:")
print(NUMERICAL_FEATURES)

print("\nBinary features:")
print(BINARY_FEATURES)

print("\nCategorical features:")
print(CATEGORICAL_FEATURES)

The feature groups are defined centrally in `00_project_config`.

This avoids duplicating feature definitions across training,
evaluation, and inference workflows.

The feature groups represent modeling decisions:

• Numerical features → imputation + standardization
• Categorical features → one-hot encoding
• Binary features → passthrough


Feature groups:

``` text 

3 Continuous Numerical Features
+
1 Binary Feature
+
15 Categorical Features
=
19 Original Model Features

```


##### 8. Inspect Missing Values

In [0]:
missing_values = df.isnull().sum()

missing_values[missing_values > 0]


TotalCharges contains 11 missing values.

Instead of manually dropping these customers, the preprocessing pipeline will use median imputation.

The median will be learned from the training dataset only to avoid data leakage.


##### 9. Train / Validation / Test Split

In [0]:
#import
from sklearn.model_selection import train_test_split


#Create Training and Temporary Sets
X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y,
)

#Split Temporary into Validation and Test
X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=VALIDATION_TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y_temp,
)

##### 10. Verify Split Sizes

In [0]:
#Verify the Shapes
print("Training:")
print("X_train:", X_train.shape)
print("y_train:", y_train.shape)

print("\nValidation:")
print("X_val:", X_val.shape)
print("y_val:", y_val.shape)

print("\nTest:")
print("X_test:", X_test.shape)
print("y_test:", y_test.shape)


##### 11. Verify Stratification

In [0]:
#Verify Stratification
print("Overall:")
print(y.value_counts(normalize=True))

print("\nTraining:")
print(y_train.value_counts(normalize=True))

print("\nValidation:")
print(y_val.value_counts(normalize=True))

print("\nTest:")
print(y_test.value_counts(normalize=True))


`stratify` preserves approximately the same target distribution across training, validation, and test datasets.

This is especially useful when the target classes are imbalanced.


##### 12. Build Reusable Preprocessor

In [0]:
import os
import sys

print("Current working directory:")
print(os.getcwd())

print("\nPython path:")
for path in sys.path:
    print(path)

In [0]:
%sh
pwd
find . -maxdepth 3 -type f | sort

In [0]:
from src.preprocessing import (
    build_preprocessor,
    validate_feature_columns,
)

In [0]:
required_features = (
    NUMERICAL_FEATURES
    + BINARY_FEATURES
    + CATEGORICAL_FEATURES
)

validate_feature_columns(
    X,
    required_features,
)


preprocessor = build_preprocessor(
    numerical_features=NUMERICAL_FEATURES,
    categorical_features=CATEGORICAL_FEATURES,
    binary_features=BINARY_FEATURES,
)

In [0]:
#Pipeline means perform multiple operations sequentially on the same columns.
#ColumnTransformer means apply different preprocessing operations to different groups of columns.

##### 13. Preprocessing Design

Numerical Features

``` text

tenure
MonthlyCharges
TotalCharges
       ↓
Median Imputation
       ↓
StandardScaler

```

Categorical Features

```text

gender
Partner
Contract
...
       ↓
OneHotEncoder

```

Binary Features

``` text

SeniorCitizen
       ↓
Passthrough

```

The reusable implementation of this preprocessing design lives in src/preprocessing.py. Notebook 02 orchestrates and verifies the preprocessing rather than redefining it.


`sparse_output=False` requests a normal dense numerical array.

One-hot encoded matrices often contain many zeros.

A sparse matrix stores mostly the non-zero values and their locations to reduce memory use.

Because the Telco dataset is small and produces only 45 final features, a dense array is convenient and appropriate for conversion to PyTorch tensors.


ColumnTransformer applies different preprocessing to different feature groups.

- Numerical
→ Imputation
→ Standardization

- Categorical
→ One-Hot Encoding

- Binary
→ Passthrough

##### 14. Fit and Transform Training Data

In [0]:
#Now comes fit_transform()
X_train_processed = preprocessor.fit_transform(
    X_train
)


`fit_transform(X_train)` performs two operations:

FIT

- Learn training median
- Learn training means
- Learn training standard deviations
- Learn training categorical values

TRANSFORM

- Impute missing values
- Standardize numerical features
- One-hot encode categorical features
- Preserve binary features

##### 15. Transform Validation and Test Data

In [0]:
#Validation and Test — Transform Only
X_val_processed = preprocessor.transform(
    X_val
)

X_test_processed = preprocessor.transform(
    X_test
)


Validation and test datasets use:

transform()

not:

fit_transform()

The preprocessing rules must be learned from training data only.

This prevents data leakage.

##### 16. Verify Processed Shapes

In [0]:
print(
    "X_train_processed:",
    X_train_processed.shape,
)

print(
    "X_val_processed:",
    X_val_processed.shape,
)

print(
    "X_test_processed:",
    X_test_processed.shape,
)


The number of rows remains unchanged.

The number of features increased:

19 original features
      ↓
One-Hot Encoding
      ↓
45 processed numerical features

##### 17. Verify Missing Values

In [0]:
import numpy as np

print(
    "Train NaNs:",
    np.isnan(X_train_processed).sum(),
)

print(
    "Validation NaNs:",
    np.isnan(X_val_processed).sum(),
)

print(
    "Test NaNs:",
    np.isnan(X_test_processed).sum(),
)


The numerical imputer successfully handled the missing TotalCharges values.

The processed matrices are now fully numerical and contain no missing values.

##### 18. Retrieve Processed Feature Names

In [0]:
feature_names = (
    preprocessor
    .get_feature_names_out()
)

print(len(feature_names))
print(feature_names)


`get_feature_names_out()` returns the names of the features produced by the fitted preprocessing pipeline.

The prefixes come from the ColumnTransformer branch names:

num__
→ numerical pipeline

cat__
→ categorical pipeline

binary__
→ binary passthrough

##### 19. Inspect One Processed Customer

In [0]:
import pandas as pd

X_train_processed_df = pd.DataFrame(
    X_train_processed,
    columns=feature_names,
    index=X_train.index,
)

In [0]:
display(
    X_train.iloc[[0]]
)

In [0]:
display(
    X_train_processed_df.iloc[[0]]
)


Example processed numerical values:

num__tenure = -1.1147

num__MonthlyCharges = 0.5043

num__TotalCharges = -0.8379

Standardized values can be:

- Negative
- Zero
- Between 0 and 1
- Greater than 1

Negative standardized values do not mean the original business value was negative.

They indicate that the original value was below the training-set mean.

 The next step is where it will cross the bridge from scikit-learn preprocessing into PyTorch: NumPy arrays → PyTorch tensors.

###### 20. Import PyTorch

In [0]:
# Import PyTorch
import torch

print(torch.__version__)


The `+cpu` suffix means this PyTorch installation is using the CPU build.

GPU acceleration is not required for this small Telco neural-network learning project.

##### 21. Convert Feature Arrays to PyTorch Tensors

In [0]:
# Convert X to tensors

X_train_tensor = torch.tensor(
    X_train_processed,
    dtype=torch.float32,
)

X_val_tensor = torch.tensor(
    X_val_processed,
    dtype=torch.float32,
)

X_test_tensor = torch.tensor(
    X_test_processed,
    dtype=torch.float32,
)


The processed feature matrices were NumPy arrays.

PyTorch neural networks operate primarily with tensors.

The transformation is:

NumPy ndarray
      ↓
torch.tensor()
      ↓
PyTorch Tensor

##### 22. Convert Targets to PyTorch Tensors

In [0]:
y_train_tensor = torch.tensor(
    y_train.to_numpy(),
    dtype=torch.float32,
).unsqueeze(1)

y_val_tensor = torch.tensor(
    y_val.to_numpy(),
    dtype=torch.float32,
).unsqueeze(1)

y_test_tensor = torch.tensor(
    y_test.to_numpy(),
    dtype=torch.float32,
).unsqueeze(1)


The target uses float32 because the planned binary-classification loss function:

BCEWithLogitsLoss

expects floating-point targets.

The target representation remains:

0.0 → No Churn

1.0 → Churn

##### 23. Why unsqueeze(1)?


Before:

y_train_tensor

Shape:

[4930]

Conceptually:

[0, 1, 0, 1, ...]


After:

unsqueeze(1)

Shape:

[4930, 1]

Conceptually:

[
 [0],
 [1],
 [0],
 [1],
 ...
]

The values do not change.

Only the tensor shape changes.

This matches the planned neural-network output shape:

[batch_size, 1]

##### 24. Verify Tensor Types

In [0]:
print(type(X_train_tensor))
print(type(y_train_tensor))

In [0]:
print(type(X_train_processed))
print(type(y_train))

##### 25. Verify Tensor Shapes

In [0]:
# Inspect the shapes
print(
    "X_train_tensor:",
    X_train_tensor.shape,
)

print(
    "y_train_tensor:",
    y_train_tensor.shape,
)


print(
    "X_val_tensor:",
    X_val_tensor.shape,
)

print(
    "y_val_tensor:",
    y_val_tensor.shape,
)


print(
    "X_test_tensor:",
    X_test_tensor.shape,
)

print(
    "y_test_tensor:",
    y_test_tensor.shape,
)

##### 26. Inspect One Tensor Record

In [0]:
print(
    X_train_tensor[0]
)

print(
    y_train_tensor[0]
)

##### 27. Create TensorDataset Objects

In [0]:
from torch.utils.data import (
    TensorDataset,
    DataLoader,
)

In [0]:

# Create TensorDataset objects
train_dataset = TensorDataset(
    X_train_tensor,
    y_train_tensor,
)

val_dataset = TensorDataset(
    X_val_tensor,
    y_val_tensor,
)

test_dataset = TensorDataset(
    X_test_tensor,
    y_test_tensor,
)



TensorDataset pairs each customer's feature tensor with the corresponding target tensor.

Example:

X_train_tensor[0]
        ↕
y_train_tensor[0]

This ensures every customer's 45 features remain associated with the correct churn label.

##### 28. Verify Dataset Sizes

In [0]:
print(
    "Train dataset:",
    len(train_dataset),
)

print(
    "Validation dataset:",
    len(val_dataset),
)

print(
    "Test dataset:",
    len(test_dataset),
)

##### 29. Inspect One TensorDataset Record

In [0]:
sample_X, sample_y = train_dataset[0]

print("Features:")
print(sample_X)

print("\nTarget:")
print(sample_y)


TensorDataset returns two items:

Features
+
Target

This is why the future training loop can use:

for X_batch, y_batch in train_loader:


Batch Size is a hyperparameter.

Batch Size = 32 means the model will process up to 32 customers before performing one parameter-update step.

##### 30. Create DataLoader Objects

In [0]:

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
)


Training:

shuffle=True

Training examples are shuffled between epochs.


Validation:

shuffle=False


Test:

shuffle=False

Validation and test datasets are used only for evaluation, so shuffling is unnecessary.

##### 31. Verify Number of Batches

In [0]:
print(
    "Training batches:",
    len(train_loader),
)

print(
    "Validation batches:",
    len(val_loader),
)

print(
    "Test batches:",
    len(test_loader),
)


Training contains:

4930 customers

Batch Size:

32

Therefore:

4930 / 32 ≈ 154.06

This requires 155 batches.

The final training batch contains fewer than 32 customers.

##### 32. Inspect One Training Batch

In [0]:
X_batch, y_batch = next(
    iter(train_loader)
)

In [0]:
print(
    "X_batch shape:",
    X_batch.shape,
)

print(
    "y_batch shape:",
    y_batch.shape,
)


This means:

X_batch

32 customers
×
45 input features


y_batch

32 customers
×
1 churn target

##### 33. Verify Batch Dtypes

In [0]:
print(
    "X_batch dtype:",
    X_batch.dtype,
)

print(
    "y_batch dtype:",
    y_batch.dtype,
)

##### 34. Determine Neural Network Input Size

In [0]:
input_size = X_train_tensor.shape[1]

print(
    "Neural network input size:",
    input_size,
)

##### 35. Connection to the Training Loop


The DataLoader now provides the exact variables used during neural-network training:

for X_batch, y_batch in train_loader:

    optimizer.zero_grad()

    outputs = model(X_batch)

    loss = criterion(
        outputs,
        y_batch
    )

    loss.backward()

    optimizer.step()


The data-preparation workflow is now directly connected to the neural-network training workflow learned in Notebook 01.

##### Key Learnings


1. Neural networks require numerical input features.

2. The Telco Gold table was converted from Spark to Pandas for preprocessing.

3. X contains model input features.

4. y contains the actual Churn_Flag target.

5. customerID was excluded because it is an identifier.

6. Churn and Churn_Flag were excluded from X because they represent the target.

7. Tenure_Group was excluded because it is derived from tenure.

8. The final model contains 19 original input features before preprocessing.

9. Continuous numerical features are:

   - tenure
   - MonthlyCharges
   - TotalCharges

10. SeniorCitizen is treated as a binary 0/1 feature.

11. Categorical features are converted into numerical form using OneHotEncoder.

12. TotalCharges contained 11 missing values.

13. Missing numerical values are handled using median imputation.

14. Numerical features are standardized using StandardScaler.

15. StandardScaler output can contain negative, zero, fractional, and values greater than 1.

16. One-hot encoded categorical features typically contain 0 or 1.

17. Binary features can pass through unchanged.

18. Data was split into:

   - 70% Training
   - 15% Validation
   - 15% Test

19. Stratification preserved approximately the same churn distribution across all datasets.

20. Preprocessing was fitted only on X_train to prevent data leakage.

21. Training data uses fit_transform().

22. Validation and test data use transform() only.

23. One-hot encoding expanded the original 19 features into 45 numerical features.

24. get_feature_names_out() retrieves the names of the processed features.

25. The scikit-learn preprocessor returned NumPy arrays.

26. NumPy arrays were converted into PyTorch tensors.

27. float32 is used for neural-network inputs and binary targets.

28. unsqueeze(1) changed target shape from:    [N]    to:    [N, 1]

29. TensorDataset pairs input features with their corresponding churn labels.

30. DataLoader creates batches for training and evaluation.

31. Batch size is a hyperparameter.

32. Training DataLoader uses shuffle=True.

33. Validation and test DataLoaders use shuffle=False.

34. Final training batch shape is:

   X_batch → [32, 45]

   y_batch → [32, 1]

35. The prepared data is now ready to enter a PyTorch neural network.

##### Conclusion


This notebook completed the full data-preparation workflow required for neural-network training.

The transformation was:

``` text 


Gold Delta Table
      ↓
Spark DataFrame
      ↓
Pandas DataFrame
      ↓
X and y
      ↓
Train / Validation / Test Split
      ↓
Numerical Preprocessing
      │
      ├── Median Imputation
      └── StandardScaler
      ↓
Categorical Preprocessing
      │
      └── OneHotEncoder
      ↓
Binary Passthrough
      ↓
45 Numerical Features
      ↓
NumPy Arrays
      ↓
PyTorch Tensors
      ↓
TensorDataset
      ↓
DataLoader
      ↓
Batches

```

Final training batch:

X_batch → [32, 45]

y_batch → [32, 1]

The data is now fully prepared for building and training the Telco Churn neural network.

##### Next Notebook


##### 03 - Build Neural Network

The next notebook will define the first PyTorch neural-network architecture.

Topics will include:

- nn.Module
- Neural-network class
- __init__()
- forward()
- nn.Linear
- Input layer
- Hidden layers
- Output layer
- ReLU activation
- Model architecture
- Input size
- Hidden-layer size
- Weights
- Biases
- Trainable parameters
- model.parameters()
- Model summary
- Forward-pass verification

The neural network will receive:

45 prepared input features

and produce:

1 binary churn output